# Peak-hour traffic: run the pipeline on Colab

**Before you start:** choose **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.
Edit only the settings cell (step 3). Start with a 2-minute test (`LIMIT_SECONDS = 120`), then set it to `0` for the full video.

If Colab disconnects during a long run, reconnect and run the same cells again. Finished chunks are skipped.

### 1. Check the GPU

In [ ]:
!nvidia-smi || echo "No GPU found. Use Runtime > Change runtime type > T4 GPU"


### 2. Connect Google Drive (videos and results live here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### 3. Settings (edit this cell)
`LINE` is `x1,y1,x2,y2` as fractions of the frame width and height. `0,0.5,1,0.5` is a horizontal line across the middle.
Use `BIN_MINUTES = 1` for short tests and `15` for full-length videos.

In [ ]:
REPO_URL = "https://github.com/vireakbothkim/peak-hour-traffic.git"   # your team's repo

DATA_DIR = "/content/drive/MyDrive/traffic"        # shared Drive folder
VIDEO    = f"{DATA_DIR}/videos/cam1.mp4"           # one .mp4 file (convert UA-DETRAC folders with frames_to_video.py)
OUT_DIR  = f"{DATA_DIR}/outputs"

LINE          = "0,0.5,1,0.5"   # counting line, see the preview below
STRIDE        = 3               # process every 3rd frame
LIMIT_SECONDS = 120             # 0 = process the whole video
BIN_MINUTES   = 1               # 15 for full-length videos
START_TIME    = ""              # clock time the recording started, e.g. "2026-11-10 06:00" (optional)
DEMO_START_S  = 0               # where the demo clip starts (seconds)


### 4. Get the code and install the libraries

In [ ]:
# first run clones the repo, later runs pull the latest version
!test -d /content/repo && git -C /content/repo pull || git clone {REPO_URL} /content/repo
%cd /content/repo
!pip install -q -r requirements.txt


### 5. Check the counting line
The red line is where vehicles are counted. **A** and **B** are the two sides: `A_to_B` means a vehicle moved from A to B.
Change `LINE` in step 3 until the line crosses the road cleanly, then run this cell again.

In [ ]:
!python src/preview_line.py --video "{VIDEO}" --line "{LINE}" --time-s 5 --out /content/preview.png
from IPython.display import Image, display
display(Image("/content/preview.png"))


### 6. Detect, track and count
This is the slow step. Results are saved to Drive chunk by chunk, so it can resume.

In [ ]:
!python src/detect_track.py --video "{VIDEO}" --out-dir "{OUT_DIR}" --line "{LINE}" --stride {STRIDE} --limit-seconds {LIMIT_SECONDS}


### 7. Group the counts into time bins and find the peak

In [ ]:
import os
NAME = os.path.splitext(os.path.basename(VIDEO))[0]
CROSSINGS = f"{OUT_DIR}/{NAME}_crossings.csv"
BINS = f"{OUT_DIR}/{NAME}_bins.csv"
START_ARG = f'--start-time "{START_TIME}"' if START_TIME else ""

!python src/aggregate.py --crossings "{CROSSINGS}" --bin-minutes {BIN_MINUTES} {START_ARG} --out "{BINS}"


### 8. Peak-hour chart

In [ ]:
CHART = f"{OUT_DIR}/{NAME}_chart.png"
!python src/plots.py --bins "{BINS}" --out "{CHART}"
display(Image(CHART))


### 9. (Optional) Annotated demo clip
Boxes, track IDs, the counting line and live counts, for the demo or as a backup recording.

In [ ]:
DEMO = f"{OUT_DIR}/{NAME}_demo.mp4"
!python src/make_demo.py --video "{VIDEO}" --line "{LINE}" --start-s {DEMO_START_S} --seconds 20 --out "{DEMO}"
print("Saved to Drive:", DEMO)
